# Warehouse integrity: stale tables, future dates, link limits, retention by period

Read-only checks to run after a pipeline re-run, before any analysis:

1. **Stale tables.** Every derived table (daily measurements, watersheds, land cover, climate, analysis dataset) must only reference stations, dams and measurements in the current core tables. A miss means the table was left from an earlier snapping or linking run.
2. **Future sample dates.** Should be 0 after `clean_wq_results` rule 9c. The raw rows show where they came from.
3. **Link distance limits.** What `exposure: max_downstream_km / max_upstream_km` keeps and drops.
4. **Raw-to-cleaned retention by period.** Cleaning that drops a different share of rows before vs after removal can create an apparent change. Compared per parameter, for linked stations only.

Uses an in-memory DuckDB over the Parquet files (`analysis/audit_utils.connect`), so `warehouse.duckdb` is never locked.

In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from analysis import audit_utils as au
from etl.build_views import DEFAULT_MIN_SAMPLES, DEFAULT_MIN_YEARS
from etl.common import wq_rules
from etl.common.config import load_config
from etl.common.warehouse import DERIVED_TABLES, stale_reference_errors, warehouse_dir

DATA_ROOT = Path(os.environ.get("DAM_REMOVAL_PROJECT_ROOT", ROOT))
CASE_STUDY = "all_removals"
cfg = load_config(CASE_STUDY, project_root=DATA_ROOT)
WAREHOUSE = warehouse_dir(cfg)

window = cfg.get("wq_window", {}) or {}
YEARS_BEFORE = int(window.get("years_before", 3))
YEARS_AFTER = int(window.get("years_after", 3))
eligibility = cfg.get("eligibility", {}) or {}
exposure = cfg.get("exposure", {}) or {}
MAX_DOWNSTREAM_KM = exposure.get("max_downstream_km")
MAX_UPSTREAM_KM = exposure.get("max_upstream_km")

con, VIEWS = au.connect(WAREHOUSE, YEARS_BEFORE, YEARS_AFTER,
                        int(eligibility.get("min_samples", DEFAULT_MIN_SAMPLES)),
                        int(eligibility.get("min_years", DEFAULT_MIN_YEARS)),
                        max_downstream_km=MAX_DOWNSTREAM_KM, max_upstream_km=MAX_UPSTREAM_KM)
print(f"{cfg.name}: window -{YEARS_BEFORE}/+{YEARS_AFTER}; link limits downstream "
      f"{MAX_DOWNSTREAM_KM or 'none'} km, upstream {MAX_UPSTREAM_KM or 'none'} km")

## 1. Stale tables

Every row should show 0. A non-zero count names the stage to re-run (in pipeline order). `build_analysis_dataset.py` refuses to run on stale context tables; `build_views.py` only reports them, since the stages that rebuild them run after it.

In [ ]:
rows = []
for table, stage in DERIVED_TABLES.items():
    if table not in VIEWS:
        rows.append({"table": table, "present": False, "stage": stage})
        continue
    n = con.sql(f"SELECT count(DISTINCT station_id) FROM {table}").fetchone()[0]
    missing = con.sql(f"""SELECT count(DISTINCT station_id) FROM {table}
                          WHERE station_id NOT IN (SELECT station_id FROM wq_station)""").fetchone()[0]
    rows.append({"table": table, "present": True, "stations": n, "stations_not_in_wq_station": missing,
                 "stage": stage})
display(pd.DataFrame(rows))

stale = stale_reference_errors(con)
print("All derived tables match the core tables" if not stale else "STALE:\n  - " + "\n  - ".join(stale))

## 2. Future sample dates

`wq_measurement` should have none (rule 9c drops them, and validation refuses them). The raw download keeps every row, so it shows which organizations and visits had a mistyped date.

In [ ]:
print("wq_measurement rows dated after today:",
      con.sql("SELECT count(*) FROM wq_measurement WHERE sample_date > current_date").fetchone()[0])
if "wq_measurement_raw" in VIEWS:
    display(con.sql("""
        SELECT OrganizationIdentifier, MonitoringLocationIdentifier, ActivityStartDate, count(*) AS rows
        FROM wq_measurement_raw
        WHERE try_cast(ActivityStartDate AS DATE) > current_date
        GROUP BY ALL ORDER BY rows DESC LIMIT 20
    """).df())

## 3. Link distance limits

`removal_link_all` is every dam-station link with a removal year; `removal_link` keeps those within the limits, and eligibility, exposure and the analysis dataset are built on it. `dam_station_link` itself is never filtered.

In [ ]:
def link_counts(view):
    return con.sql(f"""
        SELECT coalesce(side, 'unresolved') AS side, count(*) AS links, count(DISTINCT AR_ID) AS dams,
               count(DISTINCT station_id) AS stations,
               round(median(network_distance_km), 1) AS median_km, round(max(network_distance_km), 1) AS max_km
        FROM {view} GROUP BY ALL ORDER BY side
    """).df().assign(view=view)


display(pd.concat([link_counts("removal_link_all"), link_counts("removal_link")]).set_index(["view", "side"]))
print("Eligible pairs (any parameter) within the limits:",
      con.sql("SELECT count(DISTINCT (AR_ID, station_id)) FROM eligibility_pairs WHERE has_both").fetchone()[0])

## 4. Raw-to-cleaned retention by period

For each mapped WQP characteristic: raw rows downloaded vs rows kept by `clean_wq_results`, split by period relative to each linked removal (pre / post, removal year and outside excluded). Restricting to linked stations makes raw and kept rows comparable: the raw download also includes stations that were never linked.

How to read it: `pct_kept` should be similar before and after. A gap of more than ~10 points means cleaning removes more data on one side (e.g. a provider whose rows fail a rule started or stopped sampling), and that parameter's before/after comparison deserves a closer look (drop counts per rule are in the `clean_wq_results` log).

In [ ]:
con.register("mapped_parameters", pd.DataFrame({"parameter": list(wq_rules.PARAMETERS)}))
retention = con.sql(f"""
    WITH links AS (SELECT DISTINCT station_id, removal_year FROM removal_link),
    raw AS (
        SELECT MonitoringLocationIdentifier AS station_id, CharacteristicName AS parameter,
               year(try_cast(ActivityStartDate AS DATE)) AS year, count(*) AS n
        FROM wq_measurement_raw
        WHERE CharacteristicName IN (SELECT parameter FROM mapped_parameters)
        GROUP BY ALL
    ),
    kept AS (
        SELECT station_id, parameter, year(sample_date) AS year, count(*) AS n
        FROM wq_measurement GROUP BY ALL
    ),
    joined AS (
        SELECT coalesce(r.station_id, k.station_id) AS station_id,
               coalesce(r.parameter, k.parameter) AS parameter,
               coalesce(r.year, k.year) AS year,
               coalesce(r.n, 0) AS raw_rows, coalesce(k.n, 0) AS kept_rows
        FROM raw r FULL JOIN kept k
          ON r.station_id = k.station_id AND r.parameter = k.parameter AND r.year = k.year
    ),
    periods AS (
        SELECT j.*, CASE
            WHEN j.year - l.removal_year BETWEEN -{YEARS_BEFORE} AND -1 THEN 'pre'
            WHEN j.year - l.removal_year BETWEEN 1 AND {YEARS_AFTER} THEN 'post'
        END AS period
        FROM joined j JOIN links l USING (station_id)
    )
    SELECT parameter, period, sum(raw_rows) AS raw_rows, sum(kept_rows) AS kept_rows
    FROM periods WHERE period IS NOT NULL
    GROUP BY ALL
""").df()
retention["pct_kept"] = (100 * retention["kept_rows"] / retention["raw_rows"]).round(1)
table = retention.pivot(index="parameter", columns="period", values=["raw_rows", "kept_rows", "pct_kept"])
table[("pct_kept", "post - pre")] = table[("pct_kept", "post")] - table[("pct_kept", "pre")]
table.sort_values(("raw_rows", "pre"), ascending=False)